# Baseline MedGemma Evaluation — No QLoRA

This Colab notebook evaluates the **base MedGemma model without QLoRA adapters or fine-tuning** on colorectal histology image classification.

It reports and saves:
- Accuracy
- Macro F1 and weighted F1
- Per-class classification report
- Confusion matrix
- Raw predictions CSV

> Use a GPU runtime. MedGemma is gated on Hugging Face, so add your `HF_TOKEN` in Colab Secrets and accept the model license/access on Hugging Face.

## 1. Install dependencies

In [ ]:
import sys
!pip uninstall -y pillow pandas protobuf torchvision
!pip install -q -U transformers accelerate bitsandbytes datasets scikit-learn matplotlib huggingface_hub
!pip install pandas==2.2.2 Pillow==11.0.1 protobuf==3.20.3 torchvision==0.18.0
# Reinstall google-colab to ensure its dependencies are correctly set if any were removed
!pip install google-colab

Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import re
import json
import string
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from google.colab import drive, userdata
from huggingface_hub import login
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Project/output folder
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/MedGemma_Baseline_No_QLoRA_Eval'
RESULTS_DIR = os.path.join(DRIVE_PATH, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results will be saved to:", RESULTS_DIR)

Device: cuda
Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/MedGemma_Baseline_No_QLoRA_Eval/results


## 2. Imports and setup

In [ ]:
import os
import re
import json
import string
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from google.colab import drive, userdata
from huggingface_hub import login
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Project/output folder
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/MedGemma_Baseline_No_QLoRA_Eval'
RESULTS_DIR = os.path.join(DRIVE_PATH, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results will be saved to:", RESULTS_DIR)

Device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results will be saved to: /content/drive/MyDrive/MedGemma_Baseline_No_QLoRA_Eval/results


## 3. Hugging Face login

In [ ]:
try:
    login(token=userdata.get('HF_TOKEN'))
    print('Hugging Face login successful.')
except Exception as e:
    print('Hugging Face login failed. Add HF_TOKEN to Colab Secrets and accept MedGemma access.')
    raise e

Hugging Face login successful.


## 4. Dataset setup

This cell uses **CRC-VAL-HE-7K** as an external validation/test set. The classes are the standard NCT/CRC labels: `ADI, BACK, DEB, LYM, MUC, MUS, NORM, STR, TUM`.

In [ ]:
CRC_VAL_HE_7K_URL = 'https://zenodo.org/records/1214456/files/CRC-VAL-HE-7K.zip'
CRC_VAL_HE_7K_ZIP_NAME = 'CRC-VAL-HE-7K.zip'
CRC_VAL_HE_7K_DIR = 'data/CRC-VAL-HE-7K'

LABEL_NAMES = ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
label_to_id = {label: i for i, label in enumerate(LABEL_NAMES)}
id_to_label = {i: label for label, i in label_to_id.items()}

if not os.path.exists(CRC_VAL_HE_7K_DIR):
    !mkdir -p data
    !wget -q -O {CRC_VAL_HE_7K_ZIP_NAME} {CRC_VAL_HE_7K_URL}
    !unzip -q {CRC_VAL_HE_7K_ZIP_NAME} -d data/
    print('CRC-VAL-HE-7K downloaded and extracted.')
else:
    print('CRC-VAL-HE-7K already exists.')

crc_val_dataset = load_dataset('imagefolder', data_dir=CRC_VAL_HE_7K_DIR)
test_dataset = crc_val_dataset['train']
print(test_dataset)
print('Dataset labels:', test_dataset.features['label'].names)

CRC-VAL-HE-7K downloaded and extracted.


Resolving data files:   0%|          | 0/7180 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['image', 'label'],
    num_rows: 7180
})
Dataset labels: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']


## 5. Optional: choose evaluation size

Set `MAX_EVAL_SAMPLES = None` to run on all 7,180 images. For a quick test, use a smaller number like `100` or `500`.

In [ ]:
MAX_EVAL_SAMPLES = None  # Example: 100 for a quick smoke test; None for full evaluation

eval_dataset = test_dataset.shuffle(seed=SEED)
if MAX_EVAL_SAMPLES is not None:
    eval_dataset = eval_dataset.select(range(min(MAX_EVAL_SAMPLES, len(eval_dataset))))

print('Evaluation samples:', len(eval_dataset))

Evaluation samples: 7180


## 6. Load base MedGemma without QLoRA

In [ ]:
MEDGEMMA_MODEL_ID = 'google/medgemma-1.5-4b-it'

# This is inference-only quantization to fit Colab GPU memory.
# It does NOT attach QLoRA adapters and does NOT fine-tune the model.
USE_4BIT_FOR_INFERENCE = True

processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)

quantization_config = None
if USE_4BIT_FOR_INFERENCE:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

base_medgemma = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    quantization_config=quantization_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
base_medgemma.eval()
print('Base MedGemma loaded. No QLoRA adapters attached.')

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Base MedGemma loaded. No QLoRA adapters attached.


## 7. Prediction helper

In [ ]:
CLASS_PROMPT = (
    'Classify this colorectal histology patch. '
    'Choose exactly one label from: ADI, BACK, DEB, LYM, MUC, MUS, NORM, STR, TUM. '
    'Answer with only the label.'
)

def extract_label(text, valid_labels=LABEL_NAMES):
    """Extract one valid class label from MedGemma generated text."""
    text = text.upper().strip()

    # Prefer exact standalone class tokens.
    for label in valid_labels:
        if re.search(rf'\b{re.escape(label)}\b', text):
            return label

    # Fallback: first cleaned token.
    first_token = text.split()[0].translate(str.maketrans('', '', string.punctuation)) if text.split() else ''
    return first_token if first_token in valid_labels else 'UNKNOWN'

def predict_base_medgemma(image):
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': CLASS_PROMPT},
            ],
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(text=prompt, images=image, return_tensors='pt').to(base_medgemma.device)

    with torch.no_grad():
        output_ids = base_medgemma.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
        )

    decoded = processor.batch_decode(output_ids, skip_special_tokens=True)[0]

    # Keep only the generated assistant portion when possible.
    generated = decoded.split('model')[-1].strip() if 'model' in decoded else decoded.strip()
    pred_label = extract_label(generated)
    return pred_label, generated

## 8. Run baseline MedGemma evaluation

In [ ]:
true_labels = []
pred_labels = []
raw_outputs = []
image_paths = []

for i, sample in enumerate(eval_dataset):
    if i % 100 == 0:
        print(f'Processing sample {i}/{len(eval_dataset)}')

    true_label = LABEL_NAMES[sample['label']]
    pred_label, raw_text = predict_base_medgemma(sample['image'])

    true_labels.append(true_label)
    pred_labels.append(pred_label)
    raw_outputs.append(raw_text)
    image_paths.append(getattr(sample['image'], 'filename', None))

print('Done.')
print('Valid predictions:', sum(p in LABEL_NAMES for p in pred_labels), '/', len(pred_labels))
print('Unknown predictions:', sum(p == 'UNKNOWN' for p in pred_labels))

## 9. Accuracy, F1 score, and classification report

In [ ]:
# Treat UNKNOWN as wrong for overall metrics by adding it as an extra predicted class.
# For confusion matrix/class report over the 9 tissue classes, invalid predictions are filtered separately below.

y_true_all = np.array([label_to_id[x] for x in true_labels])
y_pred_all_with_unknown = np.array([label_to_id[x] if x in label_to_id else -1 for x in pred_labels])

accuracy_all = accuracy_score(y_true_all, y_pred_all_with_unknown)
macro_f1_all = f1_score(y_true_all, y_pred_all_with_unknown, labels=list(range(len(LABEL_NAMES))), average='macro', zero_division=0)
weighted_f1_all = f1_score(y_true_all, y_pred_all_with_unknown, labels=list(range(len(LABEL_NAMES))), average='weighted', zero_division=0)

valid_mask = y_pred_all_with_unknown != -1
y_true_valid = y_true_all[valid_mask]
y_pred_valid = y_pred_all_with_unknown[valid_mask]

accuracy_valid = accuracy_score(y_true_valid, y_pred_valid) if len(y_true_valid) else 0.0
macro_f1_valid = f1_score(y_true_valid, y_pred_valid, labels=list(range(len(LABEL_NAMES))), average='macro', zero_division=0) if len(y_true_valid) else 0.0
weighted_f1_valid = f1_score(y_true_valid, y_pred_valid, labels=list(range(len(LABEL_NAMES))), average='weighted', zero_division=0) if len(y_true_valid) else 0.0

metrics = {
    'model': 'base_medgemma_no_qlora',
    'num_samples': int(len(true_labels)),
    'num_valid_predictions': int(valid_mask.sum()),
    'num_unknown_predictions': int((~valid_mask).sum()),
    'accuracy_all_predictions_unknown_counted_wrong': float(accuracy_all),
    'macro_f1_all_predictions_unknown_counted_wrong': float(macro_f1_all),
    'weighted_f1_all_predictions_unknown_counted_wrong': float(weighted_f1_all),
    'accuracy_valid_predictions_only': float(accuracy_valid),
    'macro_f1_valid_predictions_only': float(macro_f1_valid),
    'weighted_f1_valid_predictions_only': float(weighted_f1_valid),
}

print(json.dumps(metrics, indent=2))

with open(os.path.join(RESULTS_DIR, 'base_medgemma_no_qlora_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

report = classification_report(
    y_true_valid,
    y_pred_valid,
    labels=list(range(len(LABEL_NAMES))),
    target_names=LABEL_NAMES,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(os.path.join(RESULTS_DIR, 'base_medgemma_no_qlora_classification_report.csv'))
report_df

## 10. Confusion matrix

In [ ]:
cm = confusion_matrix(y_true_valid, y_pred_valid, labels=list(range(len(LABEL_NAMES))))
cm_df = pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES)
cm_df.to_csv(os.path.join(RESULTS_DIR, 'base_medgemma_no_qlora_confusion_matrix.csv'))

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_NAMES)
disp.plot(ax=ax, xticks_rotation=45, values_format='d', colorbar=True)
plt.title('Base MedGemma No QLoRA Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'base_medgemma_no_qlora_confusion_matrix.png'), dpi=200)
plt.show()

## 11. Save raw predictions

In [ ]:
predictions_df = pd.DataFrame({
    'image_path': image_paths,
    'true_label': true_labels,
    'predicted_label': pred_labels,
    'raw_medgemma_output': raw_outputs,
})

predictions_path = os.path.join(RESULTS_DIR, 'base_medgemma_no_qlora_predictions.csv')
predictions_df.to_csv(predictions_path, index=False)
print('Saved predictions to:', predictions_path)
predictions_df.head()

## 12. Files created

In [ ]:
!ls -lh {RESULTS_DIR}